# Recipe Preference Matcher — MLP

Trains a tiny on-device MLP that scores how well a TheMealDB recipe matches a user's saved cuisine area and category preferences.

**Input (4 floats):**
- `area_match` — 1.0 if recipe's strArea is in user's area prefs, else 0.0
- `cat_match` — 1.0 if recipe's strCategory is in user's category prefs, else 0.0
- `area_norm` — same as area_match (extensible in future to fraction of prefs matched)
- `cat_norm` — same as cat_match

**Output:** probability in [0, 1] — higher means better match

**Label rule:** positive (1) if `area_match*60 + cat_match*40 >= 40` — i.e. any single feature match = positive

## Cell 1 — Config

In [1]:
EPOCHS = 40
BATCH_SIZE = 64
VAL_SPLIT = 0.2
N_SAMPLES = 10_000

## Cell 2 — Vocabulary Manifest

These must match the values used in the React Native app (`account.tsx` preference options).
The model itself does not depend on this list at inference time — it only needs the binary match signals.

In [2]:
# TheMealDB canonical strArea values
AREAS = [
    "American", "British", "Canadian", "Chinese", "Croatian",
    "Dutch", "Egyptian", "Filipino", "French", "Greek",
    "Indian", "Irish", "Italian", "Jamaican", "Japanese",
    "Kenyan", "Malaysian", "Mexican", "Moroccan", "Polish",
    "Portuguese", "Russian", "Spanish", "Thai", "Tunisian",
    "Turkish", "Ukrainian", "Unknown", "Vietnamese",
]

# TheMealDB canonical strCategory values
CATEGORIES = [
    "Beef", "Breakfast", "Chicken", "Dessert", "Goat",
    "Lamb", "Miscellaneous", "Pasta", "Pork", "Seafood",
    "Side", "Starter", "Vegan", "Vegetarian",
]

N_AREAS = len(AREAS)          # 29
N_CATEGORIES = len(CATEGORIES) # 14
print(f"Areas: {N_AREAS}, Categories: {N_CATEGORIES}")

Areas: 29, Categories: 14


## Cell 3 — Training Data Generation

Synthetic data: simulate random (recipe, user_prefs) pairs and label them based on match quality.

In [3]:
import numpy as np

rng = np.random.default_rng(42)

def generate_sample():
    # Simulate a recipe: one area, one category
    recipe_area_idx = rng.integers(0, N_AREAS)
    recipe_cat_idx  = rng.integers(0, N_CATEGORIES)

    # Simulate user preferences: 0–5 areas, 0–5 categories
    n_user_areas = rng.integers(0, 6)
    n_user_cats  = rng.integers(0, 6)
    user_areas   = set(rng.choice(N_AREAS, n_user_areas, replace=False).tolist())
    user_cats    = set(rng.choice(N_CATEGORIES, n_user_cats, replace=False).tolist())

    # Binary match signals
    area_match = 1.0 if recipe_area_idx in user_areas else 0.0
    cat_match  = 1.0 if recipe_cat_idx  in user_cats  else 0.0

    # 4-float input vector
    # Features 0,1: direct match signals
    # Features 2,3: same for now (extensible — could become fraction of prefs matched)
    x = np.array([area_match, cat_match, area_match, cat_match], dtype=np.float32)

    # Label: 1 if area_match*60 + cat_match*40 >= 40 (any single match = positive)
    score = area_match * 60 + cat_match * 40
    y = 1.0 if score >= 40 else 0.0

    return x, y


samples = [generate_sample() for _ in range(N_SAMPLES)]
X_train = np.stack([s[0] for s in samples])
y_train = np.array([s[1] for s in samples], dtype=np.float32)

print(f"X_train shape: {X_train.shape}")
print(f"Positive rate: {y_train.mean():.2f}")

X_train shape: (10000, 4)
Positive rate: 0.25


## Cell 4 — Class Balance Check

In [4]:
from collections import Counter

counts = Counter(y_train.astype(int))
print(f"Label distribution: {dict(counts)}")
ratio = counts[0] / max(counts[1], 1)
print(f"Negative:Positive ratio = {ratio:.1f}:1")
if ratio > 4:
    print("WARNING: class imbalance > 4:1 — consider adding class_weight to model.fit()")

Label distribution: {np.int64(1): 2487, np.int64(0): 7513}
Negative:Positive ratio = 3.0:1


## Cell 5 — Model Architecture

Tiny MLP: 4 inputs → 8 hidden (ReLU) → 1 output (Sigmoid).  
Total parameters: (4×8 + 8) + (8×1 + 1) = **49 weights** → < 5 KB TFJS shard.

In [5]:
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping

model = tf.keras.Sequential([
    layers.Dense(8, activation="relu", input_shape=(4,)),
    layers.Dense(1, activation="sigmoid"),
], name="preference_mlp")

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc")],
)

model.summary()

/home/bram/.local/lib/python3.11/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "preference_mlp"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 8)              │            40 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 49 (196.00 B)

 Trainable params: 49 (196.00 B)

 Non-trainable params: 0 (0.00 B)

## Cell 6 — Training

In [6]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True,
)

history = model.fit(
    X_train,
    y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=VAL_SPLIT,
    callbacks=[early_stopping],
)

print(f"\nFinal val_accuracy: {max(history.history['val_accuracy']):.4f}")
print(f"Final val_auc:      {max(history.history['val_auc']):.4f}")

Epoch 1/40
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9703 - auc: 0.9394 - loss: 0.5957 - val_accuracy: 1.0000 - val_auc: 1.0000 - val_loss: 0.5516
Epoch 2/40
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 882us/step - accuracy: 1.0000 - auc: 1.0000 - loss: 0.5148 - val_accuracy: 1.0000 - val_auc: 1.0000 - val_loss: 0.4860
Epoch 3/40
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 897us/step - accuracy: 1.0000 - auc: 1.0000 - loss: 0.4572 - val_accuracy: 1.0000 - val_auc: 1.0000 - val_loss: 0.4342
Epoch 4/40
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 857us/step - accuracy: 1.0000 - auc: 1.0000 - loss: 0.4093 - val_accuracy: 1.0000 - val_auc: 1.0000 - val_loss: 0.3893
Epoch 5/40
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 875us/step - accuracy: 1.0000 - auc: 1.0000 - loss: 0.3675 - val_accuracy: 1.0000 - val_auc: 1.0000 - val_loss: 0.3499
Epoch 6/40
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 886us/step - accuracy: 1.0000 - auc: 1.0000 - loss: 0.3310 - val_accuracy: 1.0000 - val_auc: 1.0000 - val_loss: 0.3156
Epoch 7/40
125/125 ━━━━━━━━━━━

## Cell 7 — Smoke Tests

Verify the model learned the correct direction. Expected:
- Both match → high score (> 0.9)
- Area-only match → medium-high (> 0.7)
- Category-only match → medium (> 0.6)
- No match → low (< 0.2)

In [7]:
test_cases = [
    ([1.0, 1.0, 1.0, 1.0], "both match    → expect HIGH  (> 0.9)"),
    ([1.0, 0.0, 1.0, 0.0], "area only     → expect MED-H (> 0.7)"),
    ([0.0, 1.0, 0.0, 1.0], "category only → expect MED   (> 0.6)"),
    ([0.0, 0.0, 0.0, 0.0], "no match      → expect LOW   (< 0.2)"),
]

for x_vals, label in test_cases:
    x = np.array([x_vals], dtype=np.float32)
    score = float(model.predict(x, verbose=0)[0][0])
    print(f"{label}  →  score = {score:.3f}")

both match    → expect HIGH  (> 0.9)  →  score = 1.000
area only     → expect MED-H (> 0.7)  →  score = 0.999
category only → expect MED   (> 0.6)  →  score = 1.000
no match      → expect LOW   (< 0.2)  →  score = 0.037


## Cell 8 — Save Keras Model

In [8]:
model.save("preference_mlp.keras")
print("Saved: preference_mlp.keras")

Saved: preference_mlp.keras


## Cell 9 — Convert to TFJS

Run this cell to produce `./tfjs_preference_mlp/model.json` and a single small `.bin` weight shard.  
Then copy the directory contents into `SpisNemt_FE/src/ml/MLP/`.

**Prerequisite:** `pip install tensorflowjs`

In [12]:
import tensorflowjs as tfjs
import os

# Create the output directory if it doesn't exist
output_path = './tmp/mlp'
if not os.path.exists(output_path):
    os.makedirs(output_path)

# Convert the model directly from the memory object
# This avoids the "InputLayer" deserialization bug in the CLI
tfjs.converters.save_keras_model(model, output_path)

print(f"Conversion complete. Files in {output_path}:")
!ls -lh {output_path}

/home/bram/.local/lib/python3.11/site-packages/tensorflow_hub/__init__.py:61: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


failed to lookup keras version from the file,
    this is likely a weight only file
Conversion complete. Files in ./tmp/mlp:
total 1.0K
-rw-r--r-- 1 bram gpuusers  196 Apr  2 15:20 group1-shard1of1.bin
-rw-r--r-- 1 bram gpuusers 3.1K Apr  2 15:20 model.json


## Cell 10 — Next Steps

After the converter runs:

1. Copy `./tfjs_preference_mlp/model.json` and `./tfjs_preference_mlp/group1-shard1of1.bin` into `SpisNemt_FE/src/ml/MLP/`
2. The TypeScript service at `SpisNemt_FE/src/services/ml/preferencesMatcher.ts` will load them via `bundleResourceIO`
3. No code changes needed — asset paths are already wired up